# Практическое занятие 2  
## HTML, web scraping и подготовка данных

### Тема занятия
**"Сбор данных из HTML-источников. Основы web scraping. Подготовка данных к дальнейшему анализу"**


## Цель занятия

Научиться извлекать данные из HTML-разметки, преобразовывать их в таблицу, очищать, сортировать, фильтровать и сохранять результат для дальнейшей работы.

После занятия слушатели должны понимать:

1. что такое HTML;
2. что такое тег, атрибут и CSS-класс;
3. что такое web scraping;
4. как из HTML получить нужные данные;
5. как преобразовать извлечённые данные в DataFrame;
6. как подготовить собранные данные к дальнейшему анализу и SQL.

# План пары

| Этап | Время | Содержание |
|---|---:|---|
| 1 | 10 минут | Повторение первой пары и постановка задачи |
| 2 | 15 минут | Введение в HTML и web scraping |
| 3 | 25 минут | Извлечение фильмов и рейтингов из HTML |
| 4 | 20 минут | Очистка, фильтрация и сортировка данных |
| 5 | 10 минут | Сохранение результата |
| 6 | 10 минут | Итоговое мини-задание и обсуждение |

---

## Связь с первой парой

На первой паре мы работали с простыми источниками данных:

- CSV;
- JSON;
- табличные данные в pandas DataFrame.

На второй паре мы разбираем ещё один источник — **HTML-страницу**.

В реальной аналитике данные не всегда лежат в аккуратной таблице. Иногда они находятся на страницах сайтов: в карточках товаров, рейтингах, списках новостей, таблицах, каталогах и т.д.

Задача аналитика — уметь извлекать такие данные и приводить их к табличному виду.

# 1. Подготовка рабочей среды

Для занятия понадобятся две библиотеки:

1. **pandas** — для работы с таблицами.
2. **BeautifulSoup** — для разбора HTML-разметки.

В Google Colab библиотека `beautifulsoup4` обычно уже установлена. Если она не установлена, её можно установить через `pip`.

In [1]:
# Импортируем pandas для работы с таблицами.
import pandas as pd

# Импортируем BeautifulSoup для разбора HTML.
from bs4 import BeautifulSoup

Проверим, что библиотеки подключились без ошибок. Если ячейка выше выполнилась успешно, можно двигаться дальше.

# 2. Введение: что такое HTML

**HTML** — это язык разметки веб-страниц.

Сайт, который мы видим в браузере, внутри состоит из HTML-элементов.

Пример HTML:

```html
<h1>Название страницы</h1>
<p>Обычный текст</p>
<a href="https://example.com">Ссылка</a>
```

Основные элементы HTML:

- **тег** — элемент разметки, например `<h1>`, `<p>`, `<div>`, `<span>`;
- **атрибут** — дополнительная информация внутри тега, например `class`, `id`, `href`;
- **текст** — содержимое между открывающим и закрывающим тегом.

Пример:

```html
<span class="rating">8.7</span>
```

Здесь:

- `span` — тег;
- `class="rating"` — атрибут;
- `8.7` — текстовое содержимое.

# 3. Что такое web scraping

**Web scraping** — это автоматизированное извлечение данных с веб-страниц.

Простая логика web scraping:

1. получить HTML-страницу;
2. найти в ней нужные элементы;
3. извлечь текст или атрибуты;
4. преобразовать результат в таблицу;
5. сохранить или проанализировать данные.

На этом занятии мы будем работать с учебной HTML-разметкой, созданной прямо в Colab.  
Это безопасно и удобно для первого знакомства.

В реальных проектах при парсинге сайтов важно учитывать:

- правила сайта;
- файл `robots.txt`;
- пользовательское соглашение;
- нагрузку на сервер;
- законность и этичность сбора данных.

# 4. Часть 1. Извлечение фильмов и рейтингов из HTML

## Постановка задачи

Представим, что у нас есть HTML-фрагмент страницы с рейтингом фильмов.

Нужно извлечь:

- название фильма;
- рейтинг фильма.

Затем нужно превратить данные в таблицу.

In [ ]:
# Создаём учебную HTML-разметку.
# В реальной задаче HTML мог бы быть получен с сайта.
# Здесь мы создаём его вручную, чтобы всем слушателям было удобно работать с одинаковыми данными.

html = '''
<html>
  <head>
    <title>Movie Ratings</title>
  </head>
  <body>
    <h1>Top Movies</h1>
    <ul class="movies">
      <li class="movie-item">
        <span class="title">The Matrix</span>
        <span class="rating">8.7</span>
        <span class="year">1999</span>
      </li>
      <li class="movie-item">
        <span class="title">Inception</span>
        <span class="rating">8.8</span>
        <span class="year">2010</span>
      </li>
      <li class="movie-item">
        <span class="title">Interstellar</span>
        <span class="rating">8.6</span>
        <span class="year">2014</span>
      </li>
      <li class="movie-item">
        <span class="title">The Godfather</span>
        <span class="rating">9.2</span>
        <span class="year">1972</span>
      </li>
      <li class="movie-item">
        <span class="title">Fight Club</span>
        <span class="rating">8.8</span>
        <span class="year">1999</span>
      </li>
    </ul>
  </body>
</html>
'''

## Задание 1. Разобрать HTML с помощью BeautifulSoup

Для работы с HTML используем объект `BeautifulSoup`.

Он превращает HTML-текст в структуру, по которой можно искать элементы.

In [ ]:
# Создаём объект BeautifulSoup.
# Второй аргумент 'html.parser' означает, что мы разбираем HTML-разметку.

soup = BeautifulSoup(html, 'html.parser')

# Выведем HTML в более читаемом виде.
print(soup.prettify())

<html>
 <head>
  <title>
   Movie Ratings
  </title>
 </head>
 <body>
  <h1>
   Top Movies
  </h1>
  <ul class="movies">
   <li class="movie-item">
    <span class="title">
     The Matrix
    </span>
    <span class="rating">
     8.7
    </span>
    <span class="year">
     1999
    </span>
   </li>
   <li class="movie-item">
    <span class="title">
     Inception
    </span>
    <span class="rating">
     8.8
    </span>
    <span class="year">
     2010
    </span>
   </li>
   <li class="movie-item">
    <span class="title">
     Interstellar
    </span>
    <span class="rating">
     8.6
    </span>
    <span class="year">
     2014
    </span>
   </li>
   <li class="movie-item">
    <span class="title">
     The Godfather
    </span>
    <span class="rating">
     9.2
    </span>
    <span class="year">
     1972
    </span>
   </li>
   <li class="movie-item">
    <span class="title">
     Fight Club
    </span>
    <span class="rating">
     8.8
    </span>
    <span class="yea

## Задание 2. Найти заголовок страницы

Сначала попробуем найти простой элемент — заголовок `<h1>`.

In [ ]:
# Метод find() находит первый элемент, который соответствует условию.

page_title = soup.find('h1')

page_title

<h1>Top Movies</h1>

In [ ]:
# Чтобы получить только текст внутри тега, используем .text

page_title_text = page_title.text

print(page_title_text)

Top Movies


## Задание 3. Найти все элементы фильмов

Каждый фильм находится внутри тега:

```html
<li class="movie-item">
```

Найдём все такие элементы.

In [ ]:
# find_all() находит все элементы, которые соответствуют условию.
# Здесь мы ищем все теги li с class='movie-item'.

movie_items = soup.find_all('li', class_='movie-item')

movie_items

[<li class="movie-item">
 <span class="title">The Matrix</span>
 <span class="rating">8.7</span>
 <span class="year">1999</span>
 </li>,
 <li class="movie-item">
 <span class="title">Inception</span>
 <span class="rating">8.8</span>
 <span class="year">2010</span>
 </li>,
 <li class="movie-item">
 <span class="title">Interstellar</span>
 <span class="rating">8.6</span>
 <span class="year">2014</span>
 </li>,
 <li class="movie-item">
 <span class="title">The Godfather</span>
 <span class="rating">9.2</span>
 <span class="year">1972</span>
 </li>,
 <li class="movie-item">
 <span class="title">Fight Club</span>
 <span class="rating">8.8</span>
 <span class="year">1999</span>
 </li>]

In [ ]:
# Посмотрим, сколько фильмов было найдено.

print('Количество найденных фильмов:', len(movie_items))

Количество найденных фильмов: 5


## Задание 4. Извлечь название первого фильма

Возьмём первый элемент из списка и достанем из него название фильма.

In [ ]:
# Берём первый элемент списка.

first_movie = movie_items[0]

# Ищем внутри него span с классом title.

first_title = first_movie.find('span', class_='title').text

print(first_title)

The Matrix


## Задание 5. Извлечь рейтинг первого фильма

Аналогично достанем рейтинг.

In [ ]:
# Ищем внутри первого фильма span с классом rating.

first_rating = first_movie.find('span', class_='rating').text

print(first_rating)

8.7


## Задание 6. Извлечь год первого фильма

В HTML также есть год выпуска фильма. Достанем его.

In [ ]:
# Ищем span с классом year.

first_year = first_movie.find('span', class_='year').text

print(first_year)

1999


# 5. Часть 2. Формирование таблицы из HTML

Теперь извлечём данные по всем фильмам.

Нам нужно получить таблицу со столбцами:

- `title`;
- `rating`;
- `year`.

In [ ]:
# Создаём пустой список.
# В него будем добавлять словари с данными по каждому фильму.

movies_data = []

# Проходим по каждому найденному HTML-элементу фильма.
for item in movie_items:
    title = item.find('span', class_='title').text
    rating = item.find('span', class_='rating').text
    year = item.find('span', class_='year').text

    # Добавляем данные в список.
    movies_data.append({
        'title': title,
        'rating': rating,
        'year': year
    })

# Посмотрим, что получилось.
movies_data

[{'title': 'The Matrix', 'rating': '8.7', 'year': '1999'},
 {'title': 'Inception', 'rating': '8.8', 'year': '2010'},
 {'title': 'Interstellar', 'rating': '8.6', 'year': '2014'},
 {'title': 'The Godfather', 'rating': '9.2', 'year': '1972'},
 {'title': 'Fight Club', 'rating': '8.8', 'year': '1999'}]

Теперь преобразуем список словарей в DataFrame.

In [ ]:
# Создаём DataFrame.

movies = pd.DataFrame(movies_data)

movies

,title,rating,year
0,The Matrix,8.7,1999
1,Inception,8.8,2010
2,Interstellar,8.6,2014
3,The Godfather,9.2,1972
4,Fight Club,8.8,1999


## Обсуждение

Мы получили таблицу, но есть важный момент: значения `rating` и `year` пока могут быть текстом, а не числами.

Перед анализом нужно проверить типы данных.

In [ ]:
# Проверяем типы данных.

movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   5 non-null      object
 1   rating  5 non-null      object
 2   year    5 non-null      object
dtypes: object(3)
memory usage: 252.0+ bytes


# 6. Часть 3. Очистка и преобразование типов данных

Сейчас столбцы `rating` и `year` могут быть распознаны как `object`, то есть текст.

Для анализа нам нужно:

- `rating` преобразовать в число с плавающей точкой;
- `year` преобразовать в целое число.

In [ ]:
# Преобразуем rating в тип float.
movies['rating'] = movies['rating'].astype(float)

# Преобразуем year в тип int.
movies['year'] = movies['year'].astype(int)

# Проверяем результат.
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   title   5 non-null      object 
 1   rating  5 non-null      float64
 2   year    5 non-null      int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 252.0+ bytes


Теперь можно выполнять числовые операции:

- считать средний рейтинг;
- находить максимальный рейтинг;
- фильтровать фильмы по рейтингу;
- сортировать фильмы.

## Задание 7. Посчитать средний рейтинг

In [ ]:
average_rating = movies['rating'].mean()

print('Средний рейтинг фильмов:', round(average_rating, 2))

Средний рейтинг фильмов: 8.82


## Задание 8. Найти фильм с максимальным рейтингом

Для этого можно отсортировать таблицу по рейтингу по убыванию и взять первую строку.

In [ ]:
top_movie = movies.sort_values(by='rating', ascending=False).head(1)

top_movie

,title,rating,year
3,The Godfather,9.2,1972


Также можно найти максимальный рейтинг отдельно.

In [ ]:
max_rating = movies['rating'].max()

print('Максимальный рейтинг:', max_rating)

Максимальный рейтинг: 9.2


## Задание 9. Отфильтровать фильмы с рейтингом выше 8.7

In [ ]:
high_rated_movies = movies[movies['rating'] > 8.7]

high_rated_movies

,title,rating,year
1,Inception,8.8,2010
3,The Godfather,9.2,1972
4,Fight Club,8.8,1999


## Задание 10. Отсортировать фильмы по рейтингу

In [ ]:
movies_sorted_by_rating = movies.sort_values(by='rating', ascending=False)

movies_sorted_by_rating

,title,rating,year
3,The Godfather,9.2,1972
4,Fight Club,8.8,1999
1,Inception,8.8,2010
0,The Matrix,8.7,1999
2,Interstellar,8.6,2014


## Задание 11. Отсортировать фильмы по году выпуска

In [ ]:
movies_sorted_by_year = movies.sort_values(by='year')

movies_sorted_by_year

,title,rating,year
3,The Godfather,9.2,1972
0,The Matrix,8.7,1999
4,Fight Club,8.8,1999
1,Inception,8.8,2010
2,Interstellar,8.6,2014


# 7. Часть 4. Сохранение результата

После извлечения и обработки данных результат часто нужно сохранить.

Сохраним таблицу фильмов в CSV-файл.

In [ ]:
# Сохраняем таблицу в CSV-файл.

movies.to_csv('movies_from_html.csv', index=False, encoding='utf-8')

print('Файл movies_from_html.csv создан успешно.')

Файл movies_from_html.csv создан успешно.


Проверим, что файл сохранился корректно.

In [ ]:
movies_check = pd.read_csv('movies_from_html.csv')

movies_check

,title,rating,year
0,The Matrix,8.7,1999
1,Inception,8.8,2010
2,Interstellar,8.6,2014
3,The Godfather,9.2,1972
4,Fight Club,8.8,1999


# 8. Часть 5. Более сложный пример: HTML-карточки товаров

Теперь рассмотрим пример чуть ближе к реальной аналитической задаче.

Представим, что есть HTML-страница с карточками товаров интернет-магазина.

Нужно извлечь:

- название товара;
- категорию;
- цену;
- наличие на складе.

In [ ]:
products_html = '''
<html>
  <body>
    <div class="product-card">
      <h2 class="product-name">Laptop Pro 14</h2>
      <span class="category">Electronics</span>
      <span class="price">1200</span>
      <span class="stock">In stock</span>
    </div>

    <div class="product-card">
      <h2 class="product-name">Wireless Mouse</h2>
      <span class="category">Electronics</span>
      <span class="price">35</span>
      <span class="stock">In stock</span>
    </div>

    <div class="product-card">
      <h2 class="product-name">Office Chair</h2>
      <span class="category">Furniture</span>
      <span class="price">180</span>
      <span class="stock">Out of stock</span>
    </div>

    <div class="product-card">
      <h2 class="product-name">Standing Desk</h2>
      <span class="category">Furniture</span>
      <span class="price">450</span>
      <span class="stock">In stock</span>
    </div>

    <div class="product-card">
      <h2 class="product-name">USB-C Hub</h2>
      <span class="category">Electronics</span>
      <span class="price">75</span>
      <span class="stock">Out of stock</span>
    </div>
  </body>
</html>
'''

## Задание 12. Разобрать HTML с товарами

In [ ]:
products_soup = BeautifulSoup(products_html, 'html.parser')

print(products_soup.prettify())

<html>
 <body>
  <div class="product-card">
   <h2 class="product-name">
    Laptop Pro 14
   </h2>
   <span class="category">
    Electronics
   </span>
   <span class="price">
    1200
   </span>
   <span class="stock">
    In stock
   </span>
  </div>
  <div class="product-card">
   <h2 class="product-name">
    Wireless Mouse
   </h2>
   <span class="category">
    Electronics
   </span>
   <span class="price">
    35
   </span>
   <span class="stock">
    In stock
   </span>
  </div>
  <div class="product-card">
   <h2 class="product-name">
    Office Chair
   </h2>
   <span class="category">
    Furniture
   </span>
   <span class="price">
    180
   </span>
   <span class="stock">
    Out of stock
   </span>
  </div>
  <div class="product-card">
   <h2 class="product-name">
    Standing Desk
   </h2>
   <span class="category">
    Furniture
   </span>
   <span class="price">
    450
   </span>
   <span class="stock">
    In stock
   </span>
  </div>
  <div class="product-card">


## Задание 13. Найти все карточки товаров

In [ ]:
product_cards = products_soup.find_all('div', class_='product-card')

print('Количество найденных товаров:', len(product_cards))

Количество найденных товаров: 5


## Задание 14. Извлечь данные по всем товарам

In [ ]:
products_data = []

for card in product_cards:
    name = card.find('h2', class_='product-name').text
    category = card.find('span', class_='category').text
    price = card.find('span', class_='price').text
    stock = card.find('span', class_='stock').text

    products_data.append({
        'name': name,
        'category': category,
        'price': price,
        'stock': stock
    })

products_data

[{'name': 'Laptop Pro 14',
  'category': 'Electronics',
  'price': '1200',
  'stock': 'In stock'},
 {'name': 'Wireless Mouse',
  'category': 'Electronics',
  'price': '35',
  'stock': 'In stock'},
 {'name': 'Office Chair',
  'category': 'Furniture',
  'price': '180',
  'stock': 'Out of stock'},
 {'name': 'Standing Desk',
  'category': 'Furniture',
  'price': '450',
  'stock': 'In stock'},
 {'name': 'USB-C Hub',
  'category': 'Electronics',
  'price': '75',
  'stock': 'Out of stock'}]

Преобразуем данные в таблицу.

In [ ]:
products = pd.DataFrame(products_data)

products

,name,category,price,stock
0,Laptop Pro 14,Electronics,1200,In stock
1,Wireless Mouse,Electronics,35,In stock
2,Office Chair,Furniture,180,Out of stock
3,Standing Desk,Furniture,450,In stock
4,USB-C Hub,Electronics,75,Out of stock


## Задание 15. Проверить типы данных и преобразовать цену

In [ ]:
products.info()

In [ ]:
# Цена сейчас может быть текстом.
# Преобразуем её в числовой тип.

products['price'] = products['price'].astype(float)

products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      5 non-null      object 
 1   category  5 non-null      object 
 2   price     5 non-null      float64
 3   stock     5 non-null      object 
dtypes: float64(1), object(3)
memory usage: 292.0+ bytes


## Задание 16. Простая аналитика по товарам

Рассчитаем:

1. количество товаров;
2. среднюю цену;
3. максимальную цену;
4. минимальную цену.

In [ ]:
print('Количество товаров:', len(products))
print('Средняя цена:', round(products['price'].mean(), 2))
print('Максимальная цена:', products['price'].max())
print('Минимальная цена:', products['price'].min())

Количество товаров: 5
Средняя цена: 388.0
Максимальная цена: 1200.0
Минимальная цена: 35.0


## Задание 17. Фильтрация товаров

Найдём только товары, которые есть в наличии.

In [ ]:
available_products = products[products['stock'] == 'In stock']

available_products

,name,category,price,stock
0,Laptop Pro 14,Electronics,1200.0,In stock
1,Wireless Mouse,Electronics,35.0,In stock
3,Standing Desk,Furniture,450.0,In stock


Найдём товары дороже 100.

In [ ]:
expensive_products = products[products['price'] > 100]

expensive_products

,name,category,price,stock
0,Laptop Pro 14,Electronics,1200.0,In stock
2,Office Chair,Furniture,180.0,Out of stock
3,Standing Desk,Furniture,450.0,In stock


## Задание 18. Группировка товаров по категориям

Посчитаем среднюю цену по каждой категории.

In [ ]:
avg_price_by_category = products.groupby('category')['price'].mean().reset_index()

avg_price_by_category = avg_price_by_category.rename(columns={'price': 'avg_price'})

avg_price_by_category

,category,avg_price
0,Electronics,436.666667
1,Furniture,315.000000


Посчитаем количество товаров по каждой категории.

In [ ]:
products_count_by_category = products.groupby('category')['name'].count().reset_index()

products_count_by_category = products_count_by_category.rename(columns={'name': 'products_count'})

products_count_by_category

,category,products_count
0,Electronics,3
1,Furniture,2


## Задание 19. Сохранить таблицу товаров

In [ ]:
products.to_csv('products_from_html.csv', index=False, encoding='utf-8')

print('Файл products_from_html.csv создан успешно.')

Файл products_from_html.csv создан успешно.


Проверим сохранённый файл.

In [ ]:
products_check = pd.read_csv('products_from_html.csv')

products_check

,name,category,price,stock
0,Laptop Pro 14,Electronics,1200.0,In stock
1,Wireless Mouse,Electronics,35.0,In stock
2,Office Chair,Furniture,180.0,Out of stock
3,Standing Desk,Furniture,450.0,In stock
4,USB-C Hub,Electronics,75.0,Out of stock


# 9. Мини-задание для самостоятельного закрепления

Используя таблицу `products`, выполните действия:

1. Выведите только столбцы `name`, `price`, `stock`.
2. Найдите товары из категории `Electronics`.
3. Найдите товары, которые есть в наличии и стоят дороже 50.
4. Отсортируйте товары по цене по убыванию.
5. Посчитайте среднюю цену только для товаров в наличии.
6. Сохраните товары в наличии в файл `available_products.csv`.

## Решение мини-задания

Ниже приведено подробное решение.

In [ ]:
# 1. Выводим только нужные столбцы.

products[['name', 'price', 'stock']]

,name,price,stock
0,Laptop Pro 14,1200.0,In stock
1,Wireless Mouse,35.0,In stock
2,Office Chair,180.0,Out of stock
3,Standing Desk,450.0,In stock
4,USB-C Hub,75.0,Out of stock


In [ ]:
# 2. Находим товары из категории Electronics.

electronics = products[products['category'] == 'Electronics']

electronics

,name,category,price,stock
0,Laptop Pro 14,Electronics,1200.0,In stock
1,Wireless Mouse,Electronics,35.0,In stock
4,USB-C Hub,Electronics,75.0,Out of stock


In [ ]:
# 3. Находим товары, которые есть в наличии и стоят дороже 50.

available_expensive = products[
    (products['stock'] == 'In stock') &
    (products['price'] > 50)
]

available_expensive

,name,category,price,stock
0,Laptop Pro 14,Electronics,1200.0,In stock
3,Standing Desk,Furniture,450.0,In stock


In [ ]:
# 4. Сортируем товары по цене по убыванию.

products_sorted = products.sort_values(by='price', ascending=False)

products_sorted

,name,category,price,stock
0,Laptop Pro 14,Electronics,1200.0,In stock
3,Standing Desk,Furniture,450.0,In stock
2,Office Chair,Furniture,180.0,Out of stock
4,USB-C Hub,Electronics,75.0,Out of stock
1,Wireless Mouse,Electronics,35.0,In stock


In [ ]:
# 5. Считаем среднюю цену только для товаров в наличии.

avg_available_price = products[products['stock'] == 'In stock']['price'].mean()

print('Средняя цена товаров в наличии:', round(avg_available_price, 2))

Средняя цена товаров в наличии: 561.67


In [ ]:
# 6. Сохраняем товары в наличии в CSV.

available_products.to_csv('available_products.csv', index=False, encoding='utf-8')

print('Файл available_products.csv создан успешно.')

Файл available_products.csv создан успешно.


# 10. Контрольные вопросы по второй паре

Ответьте на вопросы:

1. Что такое HTML?
2. Что такое тег?
3. Что такое атрибут?
4. Что такое CSS-класс?
5. Что такое web scraping?
6. Для чего используется BeautifulSoup?
7. Чем метод `find()` отличается от `find_all()`?
8. Почему после извлечения данных из HTML нужно проверять типы данных?
9. Зачем преобразовывать цену или рейтинг из текста в число?
10. Почему web scraping требует аккуратности с точки зрения правил сайта?
11. Как собранные из HTML данные можно подготовить к SQL?

# 11. Итог пары

На второй паре мы научились извлекать данные из HTML-разметки и преобразовывать их в таблицу.

Мы изучили:

- базовую структуру HTML;
- теги и CSS-классы;
- работу с BeautifulSoup;
- поиск элементов через `find()` и `find_all()`;
- извлечение текста из HTML;
- преобразование данных в DataFrame;
- очистку и преобразование типов данных;
- фильтрацию, сортировку и группировку;
- сохранение результата в CSV.

## Связь со следующими парами

На третьей паре мы перейдём к SQL.

Данные, которые мы подготовили на первых двух парах, уже имеют табличный вид.  
Следующий шаг — научиться загружать такие данные в базу данных и выполнять SQL-запросы:

- `SELECT`;
- `WHERE`;
- `ORDER BY`;
- `GROUP BY`;
- агрегатные функции.

Именно это будет основой следующего занятия.